In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from PIL import Image
import numpy as np

# Define the Final Object Detection Model
class ObjectDetectionModel(tf.keras.Model):
    def __init__(self):
        super(ObjectDetectionModel, self).__init__()
        # Backbone
        self.backbone = models.Sequential([
            layers.Conv2D(32, (3, 3), strides=2, padding='same'),
            layers.BatchNormalization(momentum=0.9),
            layers.ReLU(),
            layers.Conv2D(64, (3, 3), strides=2, padding='same'),
            layers.BatchNormalization(momentum=0.9),
            layers.ReLU(),
            layers.Conv2D(128, (3, 3), strides=2, padding='same'),
            layers.BatchNormalization(momentum=0.9),
            layers.ReLU(),
        ])

        # Feature Pyramid Network (Neck)
        self.fpn = layers.Conv2D(512, (1, 1), padding='same')

        # Detection Head
        self.head = models.Sequential([
            layers.Conv2D(256, (3, 3), padding='same'),
            layers.ReLU(),
            layers.Conv2D(128, (3, 3), padding='same'),
            layers.ReLU(),
            layers.Conv2D(64, (3, 3), padding='same'),
            layers.ReLU(),
            layers.Conv2D(5, (1, 1))  # Output: bounding box params + confidence
        ])

    def call(self, x):
        # Forward pass through backbone
        features = self.backbone(x)
        # Forward pass through FPN
        pyramid_features = self.fpn(features)
        # Forward pass through detection head
        predictions = self.head(pyramid_features)
        return predictions

# Preprocessing function
def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = image.resize((512, 512))
    image_array = np.array(image) / 255.0  # Normalize to [0, 1]
    return np.expand_dims(image_array, axis=0).astype(np.float32)  # Add batch dimension

# Non-Maximum Suppression (NMS)
def non_max_suppression(predictions, iou_threshold=0.5):
    filtered_predictions = []
    for pred in predictions:
        if pred[-1] > iou_threshold:  # Confidence check
            filtered_predictions.append(pred)
    return filtered_predictions

# Object Detection Pipeline
def object_detection_pipeline(image_path):
    # Preprocess the input image
    image_tensor = preprocess_image(image_path)

    # Initialize the model
    model = ObjectDetectionModel()
    model.build(input_shape=(None, 512, 512, 3))  # Build the model with input shape
    model.summary()  # Print model summary

    # Forward pass
    predictions = model(image_tensor)

    # Reshape predictions and apply NMS
    predictions_np = predictions.numpy().reshape(-1, 5)  # Reshape to (N, 5)
    bounding_boxes = non_max_suppression(predictions_np, iou_threshold=0.5)

    # Count objects
    total_count = len(bounding_boxes)
    return total_count, bounding_boxes


In [4]:
model = ObjectDetectionModel()

In [2]:
data_path = '/content/drive/MyDrive/dataset'  # Update this to the folder containing your data files

# Load the data
X = np.load(os.path.join(data_path, 'X.npy'))  # Images
Y = np.load(os.path.join(data_path, 'Y.npy'))  # Labels

# Split the data into training and testing sets (70-30 split, stratified)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, stratify=Y, random_state=42
)

# Further split training data into training and validation sets (70-30 split of training data)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.3, stratify=Y_train, random_state=42
)

# Early Stopping and Learning Rate Reduction Callbacks
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=5, min_lr=1e-6)

# Train the Model
history = model.fit(X_train, Y_train,epochs=600, batch_size=32,
                    callbacks=[early_stopping, reduce_lr])

Epoch 1/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.0990 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3027
Epoch 2/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.1015 - loss: 2.2997 - val_accuracy: 0.1025 - val_loss: 2.2997
Epoch 3/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.1019 - loss: 2.2967 - val_accuracy: 0.1029 - val_loss: 2.2967
Epoch 4/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step - accuracy: 0.1044 - loss: 2.2937 - val_accuracy: 0.1054 - val_loss: 2.2937
Epoch 5/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.1049 - loss: 2.2907 - val_accuracy: 0.1059 - val_loss: 2.2907
Epoch 6/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 55ms/step - accuracy: 0.1073 - loss: 2.2877 - val_accuracy: 0.1083 - val_loss: 2.2877
Epoch 7/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.1078 - loss: 2.2847 - val_accuracy: 0.1088 - val_loss: 2.2847
Epoch 8/800
187/187 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.1103 - loss: 2.2817 - 

In [5]:
model_path = "item_count_detection_model.h5"
model.save(model_path)
# print(f"Model saved at {model_path}")

# Download the Model
def download_model(file_path):
    from google.colab import files
    if os.path.exists(file_path):
        files.download(file_path)
    else:
        print("File not found!")